# Sales SQL Analysis Project

## Objetivo
Construir un proyecto completo de análisis de ventas usando PostgreSQL, SQL y Python.

## Stack
- PostgreSQL (Neon)
- Python (pandas + SQLAlchemy)
- Jupyter Notebook
- DBeaver

## Fases del proyecto
1. Conexión a la base de datos
2. Creación del esquema
3. Inserción de datos
4. Análisis SQL
5. Conclusiones

In [1]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()

user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
db = os.getenv("POSTGRES_DATABASE")
port = os.getenv("POSTGRES_PORT")

DATABASE_URL = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}?sslmode=require"

engine = create_engine(DATABASE_URL)

print("Conexión preparada")

Conexión preparada


In [2]:
df = pd.read_sql("SELECT version();", engine)
df

,version
0,PostgreSQL 17.8 (a48d9ca) on aarch64-unknown-l...


## Creación del esquema de la base de datos

En esta sección se crea el modelo relacional del proyecto de ventas.
Las tablas principales son:

- regions
- customers
- categories
- products
- employees
- orders
- order_items

In [7]:
schema_statements = [
    "DROP TABLE IF EXISTS order_items CASCADE;",
    "DROP TABLE IF EXISTS orders CASCADE;",
    "DROP TABLE IF EXISTS products CASCADE;",
    "DROP TABLE IF EXISTS categories CASCADE;",
    "DROP TABLE IF EXISTS customers CASCADE;",
    "DROP TABLE IF EXISTS employees CASCADE;",
    "DROP TABLE IF EXISTS regions CASCADE;",

    """
    CREATE TABLE regions (
        region_id INT PRIMARY KEY,
        region_name VARCHAR(100) NOT NULL,
        country VARCHAR(100) NOT NULL
    );
    """,

    """
    CREATE TABLE customers (
        customer_id INT PRIMARY KEY,
        customer_name VARCHAR(150) NOT NULL,
        email VARCHAR(150) UNIQUE,
        signup_date DATE NOT NULL,
        segment VARCHAR(50) NOT NULL,
        region_id INT,
        FOREIGN KEY (region_id) REFERENCES regions(region_id)
    );
    """,

    """
    CREATE TABLE categories (
        category_id INT PRIMARY KEY,
        category_name VARCHAR(100) NOT NULL
    );
    """,

    """
    CREATE TABLE products (
        product_id INT PRIMARY KEY,
        product_name VARCHAR(150) NOT NULL,
        category_id INT NOT NULL,
        unit_price DECIMAL(10,2) NOT NULL,
        cost_price DECIMAL(10,2) NOT NULL,
        launch_date DATE,
        FOREIGN KEY (category_id) REFERENCES categories(category_id)
    );
    """,

    """
    CREATE TABLE employees (
        employee_id INT PRIMARY KEY,
        employee_name VARCHAR(150) NOT NULL,
        hire_date DATE NOT NULL,
        job_title VARCHAR(100) NOT NULL,
        region_id INT,
        FOREIGN KEY (region_id) REFERENCES regions(region_id)
    );
    """,

    """
    CREATE TABLE orders (
        order_id INT PRIMARY KEY,
        customer_id INT NOT NULL,
        employee_id INT,
        order_date DATE NOT NULL,
        status VARCHAR(50) NOT NULL,
        shipping_cost DECIMAL(10,2) DEFAULT 0.00,
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
        FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
    );
    """,

    """
    CREATE TABLE order_items (
        order_item_id INT PRIMARY KEY,
        order_id INT NOT NULL,
        product_id INT NOT NULL,
        quantity INT NOT NULL CHECK (quantity > 0),
        unit_price DECIMAL(10,2) NOT NULL,
        discount DECIMAL(4,2) DEFAULT 0.00 CHECK (discount >= 0 AND discount <= 1),
        FOREIGN KEY (order_id) REFERENCES orders(order_id),
        FOREIGN KEY (product_id) REFERENCES products(product_id)
    );
    """
]

In [8]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

tables_df = pd.read_sql(query, engine)
tables_df

,table_name
0,categories
1,customers
2,employees
3,order_items
4,orders
5,products
6,regions


## Inserción de datos base

En esta fase se insertan datos iniciales en las tablas maestras:
- regiones
- categorías
- empleados

Estos datos servirán como base para poblar después clientes, productos y pedidos.

In [9]:
from sqlalchemy import text

regions_sql = """
INSERT INTO regions (region_id, region_name, country) VALUES
(1, 'Madrid', 'Spain'),
(2, 'Catalonia', 'Spain'),
(3, 'Andalusia', 'Spain'),
(4, 'Valencian Community', 'Spain'),
(5, 'Basque Country', 'Spain');
"""

with engine.begin() as conn:
    conn.execute(text(regions_sql))

print("Regions inserted successfully")

Regions inserted successfully


In [10]:
pd.read_sql("SELECT * FROM regions ORDER BY region_id;", engine)

,region_id,region_name,country
0,1,Madrid,Spain
1,2,Catalonia,Spain
2,3,Andalusia,Spain
3,4,Valencian Community,Spain
4,5,Basque Country,Spain


In [11]:
categories_sql = """
INSERT INTO categories (category_id, category_name) VALUES
(1, 'Electronics'),
(2, 'Office Supplies'),
(3, 'Furniture'),
(4, 'Accessories'),
(5, 'Software');
"""

with engine.begin() as conn:
    conn.execute(text(categories_sql))

print("Categories inserted successfully")

Categories inserted successfully


In [12]:
pd.read_sql("SELECT * FROM categories ORDER BY category_id;", engine)

,category_id,category_name
0,1,Electronics
1,2,Office Supplies
2,3,Furniture
3,4,Accessories
4,5,Software


In [13]:
employees_sql = """
INSERT INTO employees (employee_id, employee_name, hire_date, job_title, region_id) VALUES
(1, 'Laura Martinez', '2021-03-15', 'Sales Representative', 1),
(2, 'Carlos Gomez', '2020-06-10', 'Sales Representative', 2),
(3, 'Ana Ruiz', '2022-01-20', 'Account Manager', 3),
(4, 'David Navarro', '2019-11-05', 'Senior Sales Representative', 4),
(5, 'Marta Sanchez', '2023-02-12', 'Sales Representative', 5),
(6, 'Javier Torres', '2021-09-01', 'Account Manager', 1),
(7, 'Elena Moreno', '2020-04-18', 'Senior Sales Representative', 2),
(8, 'Pablo Castro', '2022-07-30', 'Sales Representative', 3);
"""

with engine.begin() as conn:
    conn.execute(text(employees_sql))

print("Employees inserted successfully")

Employees inserted successfully


In [14]:
pd.read_sql("SELECT * FROM employees ORDER BY employee_id;", engine)

,employee_id,employee_name,hire_date,job_title,region_id
0,1,Laura Martinez,2021-03-15,Sales Representative,1
1,2,Carlos Gomez,2020-06-10,Sales Representative,2
2,3,Ana Ruiz,2022-01-20,Account Manager,3
3,4,David Navarro,2019-11-05,Senior Sales Representative,4
4,5,Marta Sanchez,2023-02-12,Sales Representative,5
5,6,Javier Torres,2021-09-01,Account Manager,1
6,7,Elena Moreno,2020-04-18,Senior Sales Representative,2
7,8,Pablo Castro,2022-07-30,Sales Representative,3


In [15]:
for table in ["regions", "categories", "employees"]:
    df = pd.read_sql(f"SELECT * FROM {table};", engine)
    print(f"{table}: {len(df)} rows")

regions: 5 rows
categories: 5 rows
employees: 8 rows


## Inserción de customers y products

En esta fase se insertan datos de clientes y productos.
Estos datos permitirán realizar análisis de comportamiento de clientes y rendimiento de productos.


In [16]:
customers_sql = """
INSERT INTO customers (customer_id, customer_name, email, signup_date, segment, region_id) VALUES
(1, 'Juan Perez', 'juan.perez@email.com', '2022-01-15', 'Consumer', 1),
(2, 'Maria Lopez', 'maria.lopez@email.com', '2021-11-20', 'Corporate', 2),
(3, 'Carlos Ruiz', 'carlos.ruiz@email.com', '2023-03-05', 'Small Business', 3),
(4, 'Laura Garcia', 'laura.garcia@email.com', '2020-07-12', 'Consumer', 4),
(5, 'David Fernandez', 'david.fernandez@email.com', '2022-09-30', 'Corporate', 5),
(6, 'Elena Torres', 'elena.torres@email.com', '2021-05-18', 'Consumer', 1),
(7, 'Pablo Sanchez', 'pablo.sanchez@email.com', '2023-01-22', 'Small Business', 2),
(8, 'Marta Diaz', 'marta.diaz@email.com', '2022-06-10', 'Corporate', 3),
(9, 'Javier Romero', 'javier.romero@email.com', '2021-02-14', 'Consumer', 4),
(10, 'Ana Morales', 'ana.morales@email.com', '2023-04-01', 'Small Business', 5);
"""

In [17]:
with engine.begin() as conn:
    conn.execute(text(customers_sql))

print("Customers inserted")

Customers inserted


In [18]:
pd.read_sql("SELECT * FROM customers;", engine)

,customer_id,customer_name,email,signup_date,segment,region_id
0,1,Juan Perez,juan.perez@email.com,2022-01-15,Consumer,1
1,2,Maria Lopez,maria.lopez@email.com,2021-11-20,Corporate,2
2,3,Carlos Ruiz,carlos.ruiz@email.com,2023-03-05,Small Business,3
3,4,Laura Garcia,laura.garcia@email.com,2020-07-12,Consumer,4
4,5,David Fernandez,david.fernandez@email.com,2022-09-30,Corporate,5
5,6,Elena Torres,elena.torres@email.com,2021-05-18,Consumer,1
6,7,Pablo Sanchez,pablo.sanchez@email.com,2023-01-22,Small Business,2
7,8,Marta Diaz,marta.diaz@email.com,2022-06-10,Corporate,3
8,9,Javier Romero,javier.romero@email.com,2021-02-14,Consumer,4
9,10,Ana Morales,ana.morales@email.com,2023-04-01,Small Business,5


In [19]:
products_sql = """
INSERT INTO products (product_id, product_name, category_id, unit_price, cost_price, launch_date) VALUES
(1, 'Laptop Pro 15', 1, 1200.00, 800.00, '2021-01-10'),
(2, 'Wireless Mouse', 4, 25.00, 10.00, '2020-06-15'),
(3, 'Office Chair', 3, 180.00, 100.00, '2019-11-05'),
(4, 'Mechanical Keyboard', 1, 90.00, 50.00, '2022-02-20'),
(5, 'Notebook Pack', 2, 15.00, 5.00, '2021-09-01'),
(6, 'Monitor 27"', 1, 300.00, 180.00, '2020-03-12'),
(7, 'Desk Lamp', 3, 45.00, 20.00, '2022-07-30'),
(8, 'USB-C Hub', 4, 60.00, 25.00, '2021-12-01'),
(9, 'Project Management Software', 5, 200.00, 50.00, '2023-01-01'),
(10, 'Printer', 1, 250.00, 150.00, '2020-08-18');
"""

In [20]:
with engine.begin() as conn:
    conn.execute(text(products_sql))

print("Products inserted")

Products inserted


In [21]:
pd.read_sql("SELECT * FROM products;", engine)

,product_id,product_name,category_id,unit_price,cost_price,launch_date
0,1,Laptop Pro 15,1,1200.0,800.0,2021-01-10
1,2,Wireless Mouse,4,25.0,10.0,2020-06-15
2,3,Office Chair,3,180.0,100.0,2019-11-05
3,4,Mechanical Keyboard,1,90.0,50.0,2022-02-20
4,5,Notebook Pack,2,15.0,5.0,2021-09-01
5,6,"Monitor 27""",1,300.0,180.0,2020-03-12
6,7,Desk Lamp,3,45.0,20.0,2022-07-30
7,8,USB-C Hub,4,60.0,25.0,2021-12-01
8,9,Project Management Software,5,200.0,50.0,2023-01-01
9,10,Printer,1,250.0,150.0,2020-08-18


In [22]:
for table in ["customers", "products"]:
    df = pd.read_sql(f"SELECT * FROM {table};", engine)
    print(f"{table}: {len(df)} rows")

customers: 10 rows
products: 10 rows


## Inserción de orders y order_items

En esta fase se insertan pedidos y líneas de pedido.
Estas tablas representan las transacciones reales del negocio y permiten calcular métricas como ingresos, descuentos y beneficio.

In [25]:
orders_sql = """
INSERT INTO orders (order_id, customer_id, employee_id, order_date, status, shipping_cost) VALUES
(1, 1, 1, '2024-01-10', 'Completed', 8.50),
(2, 2, 2, '2024-01-12', 'Completed', 12.00),
(3, 3, 3, '2024-01-15', 'Completed', 6.75),
(4, 4, 4, '2024-01-18', 'Completed', 9.20),
(5, 5, 5, '2024-01-20', 'Pending', 10.00),
(6, 6, 6, '2024-01-25', 'Completed', 7.80),
(7, 7, 7, '2024-02-01', 'Completed', 11.30),
(8, 8, 8, '2024-02-03', 'Cancelled', 5.00),
(9, 9, 1, '2024-02-07', 'Completed', 8.90),
(10, 10, 2, '2024-02-10', 'Completed', 13.40),
(11, 1, 3, '2024-02-14', 'Completed', 6.00),
(12, 2, 4, '2024-02-18', 'Pending', 9.50),
(13, 3, 5, '2024-02-22', 'Completed', 7.10),
(14, 4, 6, '2024-02-25', 'Completed', 10.80),
(15, 5, 7, '2024-02-28', 'Completed', 12.60);
"""

In [26]:
with engine.begin() as conn:
    conn.execute(text(orders_sql))

print("Orders inserted")

Orders inserted


In [27]:
pd.read_sql("SELECT * FROM orders ORDER BY order_id;", engine)

,order_id,customer_id,employee_id,order_date,status,shipping_cost
0,1,1,1,2024-01-10,Completed,8.50
1,2,2,2,2024-01-12,Completed,12.00
2,3,3,3,2024-01-15,Completed,6.75
3,4,4,4,2024-01-18,Completed,9.20
4,5,5,5,2024-01-20,Pending,10.00
5,6,6,6,2024-01-25,Completed,7.80
6,7,7,7,2024-02-01,Completed,11.30
7,8,8,8,2024-02-03,Cancelled,5.00
8,9,9,1,2024-02-07,Completed,8.90
9,10,10,2,2024-02-10,Completed,13.40


In [28]:
order_items_sql = """
INSERT INTO order_items (order_item_id, order_id, product_id, quantity, unit_price, discount) VALUES
(1, 1, 1, 1, 1200.00, 0.05),
(2, 1, 2, 2, 25.00, 0.00),

(3, 2, 3, 1, 180.00, 0.10),
(4, 2, 5, 5, 15.00, 0.00),

(5, 3, 4, 1, 90.00, 0.00),
(6, 3, 8, 2, 60.00, 0.15),

(7, 4, 6, 2, 300.00, 0.05),
(8, 4, 2, 1, 25.00, 0.00),

(9, 5, 9, 1, 200.00, 0.20),

(10, 6, 7, 2, 45.00, 0.00),
(11, 6, 5, 3, 15.00, 0.05),

(12, 7, 10, 1, 250.00, 0.10),
(13, 7, 2, 3, 25.00, 0.00),

(14, 8, 1, 1, 1200.00, 0.00),

(15, 9, 3, 1, 180.00, 0.00),
(16, 9, 4, 1, 90.00, 0.05),

(17, 10, 6, 1, 300.00, 0.10),
(18, 10, 8, 1, 60.00, 0.00),

(19, 11, 5, 10, 15.00, 0.10),
(20, 11, 2, 2, 25.00, 0.00),

(21, 12, 9, 2, 200.00, 0.15),

(22, 13, 1, 1, 1200.00, 0.08),
(23, 13, 10, 1, 250.00, 0.05),

(24, 14, 7, 3, 45.00, 0.00),
(25, 14, 8, 2, 60.00, 0.10),

(26, 15, 3, 2, 180.00, 0.12),
(27, 15, 5, 4, 15.00, 0.00);
"""

In [29]:
with engine.begin() as conn:
    conn.execute(text(order_items_sql))

print("Order items inserted")

Order items inserted


In [30]:
pd.read_sql("SELECT * FROM order_items ORDER BY order_item_id;", engine)

,order_item_id,order_id,product_id,quantity,unit_price,discount
0,1,1,1,1,1200.0,0.05
1,2,1,2,2,25.0,0.00
2,3,2,3,1,180.0,0.10
3,4,2,5,5,15.0,0.00
4,5,3,4,1,90.0,0.00
5,6,3,8,2,60.0,0.15
6,7,4,6,2,300.0,0.05
7,8,4,2,1,25.0,0.00
8,9,5,9,1,200.0,0.20
9,10,6,7,2,45.0,0.00


In [31]:
for table in ["orders", "order_items"]:
    df = pd.read_sql(f"SELECT * FROM {table};", engine)
    print(f"{table}: {len(df)} rows")

orders: 15 rows
order_items: 27 rows
